# Course 03: Working with Notebooks in Vertex AI

**CareerAlign GCP Machine Learning Engineer Certification**

This notebook is the companion to the Course 03 study guide. It covers hands-on exploration of
GCP notebook environments, BigQuery integration, Cloud Storage access, custom package installation,
parameterized notebook execution, and GPU availability checks.

**Exam Section:** Section 2 — Collaborating to Manage Data and Models

---

## 1. Check Current Runtime / Environment Info

Understanding your runtime environment is the first step when working in any GCP notebook.
These checks help you verify the machine type, available resources, and installed libraries.

In [ ]:
import os
import sys
import platform
import subprocess

print("=" * 60)
print("RUNTIME ENVIRONMENT INFO")
print("=" * 60)

# Python version
print(f"Python version:  {sys.version}")
print(f"Platform:        {platform.platform()}")
print(f"Architecture:    {platform.machine()}")

# Check if running in Colab / Colab Enterprise
IN_COLAB = "google.colab" in sys.modules
print(f"\nRunning in Colab: {IN_COLAB}")

# Check GCP project
try:
    project = subprocess.check_output(
        ["gcloud", "config", "get-value", "project"],
        stderr=subprocess.DEVNULL
    ).decode().strip()
    print(f"GCP Project:     {project}")
except Exception:
    print("GCP Project:     Not configured (gcloud not available)")

# CPU info
print(f"\nCPU count:       {os.cpu_count()}")

# Memory info (Linux/Mac)
try:
    import psutil
    mem = psutil.virtual_memory()
    print(f"Total RAM:       {mem.total / (1024**3):.1f} GB")
    print(f"Available RAM:   {mem.available / (1024**3):.1f} GB")
except ImportError:
    print("Install psutil for memory info: pip install psutil")

In [ ]:
# Check installed key packages and their versions
packages_to_check = [
    "google-cloud-bigquery",
    "google-cloud-storage",
    "google-cloud-aiplatform",
    "pandas",
    "numpy",
    "tensorflow",
    "torch",
    "scikit-learn",
    "papermill",
]

print("\nInstalled Package Versions:")
print("-" * 40)
for pkg in packages_to_check:
    try:
        import importlib.metadata
        version = importlib.metadata.version(pkg)
        print(f"  {pkg:35s} {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"  {pkg:35s} NOT INSTALLED")

## 2. BigQuery Integration: Query Directly from Notebook

BigQuery is the most common data source for ML workloads on GCP. You can query data
directly from notebooks using the Python client library or cell magic.

In [ ]:
# Install if needed
# !pip install google-cloud-bigquery db-dtypes pyarrow --quiet

from google.cloud import bigquery

# Initialize client (uses default credentials in GCP notebook environments)
client = bigquery.Client()

print(f"Connected to project: {client.project}")

In [ ]:
# Query a public dataset — USA names
query = """
SELECT
    name,
    gender,
    SUM(number) AS total_count
FROM
    `bigquery-public-data.usa_names.usa_1910_2013`
GROUP BY
    name, gender
ORDER BY
    total_count DESC
LIMIT 10
"""

df_names = client.query(query).to_dataframe()
print("Top 10 most popular names in the USA (1910-2013):")
df_names

In [ ]:
# Query the penguins dataset (commonly used in ML examples)
query_penguins = """
SELECT
    species,
    island,
    culmen_length_mm,
    culmen_depth_mm,
    flipper_length_mm,
    body_mass_g,
    sex
FROM
    `bigquery-public-data.ml_datasets.penguins`
WHERE
    body_mass_g IS NOT NULL
"""

df_penguins = client.query(query_penguins).to_dataframe()
print(f"Penguins dataset: {len(df_penguins)} rows, {len(df_penguins.columns)} columns")
print(f"\nSpecies distribution:")
print(df_penguins["species"].value_counts())
df_penguins.head()

In [ ]:
# Check how much data a query will scan (dry run)
job_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)

dry_run_query = """
SELECT *
FROM `bigquery-public-data.chicago_taxi_trips.taxi_trips`
WHERE trip_start_timestamp >= '2023-01-01'
"""

query_job = client.query(dry_run_query, job_config=job_config)
bytes_processed = query_job.total_bytes_processed
print(f"This query will process: {bytes_processed / (1024**3):.2f} GB")
print(f"Estimated cost: ${bytes_processed / (1024**4) * 6.25:.4f} (on-demand pricing)")

## 3. Cloud Storage: Read/Write Files from Notebook

Cloud Storage (GCS) is the primary object store for ML artifacts — training data,
model checkpoints, evaluation results, and more.

In [ ]:
# Install if needed
# !pip install google-cloud-storage gcsfs --quiet

from google.cloud import storage
import pandas as pd

# Initialize storage client
storage_client = storage.Client()

# List buckets in your project
print("Buckets in your project:")
for bucket in storage_client.list_buckets():
    print(f"  - {bucket.name}")

In [ ]:
# Reading a CSV directly from GCS using pandas + gcsfs
# (gcsfs is pre-installed in GCP notebook environments)

# Example with a public GCS bucket
# df = pd.read_csv("gs://cloud-samples-data/ai-platform/iris/iris_data.csv",
#                   header=None,
#                   names=["sepal_length", "sepal_width", "petal_length", "petal_width", "species"])
# print(f"Loaded {len(df)} rows from GCS")
# df.head()

# --- Write a DataFrame to GCS ---
# Save our penguins data to GCS as CSV
# BUCKET_NAME = "your-bucket-name"  # <-- Replace with your bucket
# OUTPUT_PATH = f"gs://{BUCKET_NAME}/notebooks/penguins_export.csv"
# df_penguins.to_csv(OUTPUT_PATH, index=False)
# print(f"Saved {len(df_penguins)} rows to {OUTPUT_PATH}")

print("Uncomment the code above and replace BUCKET_NAME to test GCS read/write.")
print("In GCP notebook environments, authentication is handled automatically.")

In [ ]:
# Upload and download files using the storage client

def upload_to_gcs(bucket_name, source_file, destination_blob):
    """Upload a local file to GCS."""
    bucket = storage_client.bucket(bucket_name)
    blob = bucket.blob(destination_blob)
    blob.upload_from_filename(source_file)
    print(f"Uploaded {source_file} to gs://{bucket_name}/{destination_blob}")

def download_from_gcs(bucket_name, source_blob, destination_file):
    """Download a file from GCS to local."""
    bucket = storage_client.bucket(bucket_name)
    blob = bucket.blob(source_blob)
    blob.download_to_filename(destination_file)
    print(f"Downloaded gs://{bucket_name}/{source_blob} to {destination_file}")

def list_blobs(bucket_name, prefix=None):
    """List objects in a GCS bucket with optional prefix filter."""
    blobs = storage_client.list_blobs(bucket_name, prefix=prefix)
    for blob in blobs:
        print(f"  {blob.name} ({blob.size / 1024:.1f} KB)")

# Example usage (uncomment and set your bucket):
# upload_to_gcs("my-bucket", "local_file.csv", "data/uploaded_file.csv")
# download_from_gcs("my-bucket", "data/uploaded_file.csv", "downloaded_file.csv")
# list_blobs("my-bucket", prefix="data/")

print("GCS helper functions defined. Set your bucket name to use them.")

## 4. Install Custom Packages in a Notebook Environment

GCP notebooks come pre-installed with common ML libraries, but you often need
additional packages. The approach differs by environment.

In [ ]:
# Method 1: pip install in a cell (works in all environments)
# The ! prefix runs shell commands from a notebook cell

!pip install papermill --quiet
!pip install google-cloud-aiplatform --quiet

print("\nPackages installed successfully.")
print("Note: In Colab Enterprise, installs are ephemeral (lost on runtime disconnect).")
print("In Workbench, installs persist on the VM's disk.")

In [ ]:
# Method 2: Install from a requirements.txt file
# This is the preferred approach for reproducible environments

requirements_content = """# requirements.txt for ML notebook
google-cloud-bigquery>=3.0.0
google-cloud-storage>=2.0.0
google-cloud-aiplatform>=1.30.0
pandas>=2.0.0
scikit-learn>=1.3.0
matplotlib>=3.7.0
papermill>=2.4.0
"""

# Write requirements file
with open("requirements.txt", "w") as f:
    f.write(requirements_content)

print("requirements.txt created. Install with:")
print("  !pip install -r requirements.txt --quiet")

In [ ]:
# Method 3: For Workbench User-Managed — create a startup script
# This runs every time the instance starts, ensuring packages persist

startup_script = """#!/bin/bash
# Startup script for Vertex AI Workbench User-Managed Notebook
# Place in: /opt/deeplearning/bin/startup_script.sh

pip install --upgrade google-cloud-bigquery google-cloud-storage
pip install papermill nbstripout

# Configure nbstripout for version control
nbstripout --install --global

echo "Startup script completed at $(date)"
"""

print("Startup script template for Workbench User-Managed instances:")
print(startup_script)

## 5. Parameterized Notebook Execution with Papermill

Papermill allows you to pass parameters to notebooks and execute them programmatically.
This is the key pattern for moving notebooks from exploration to production.

In [ ]:
# This cell is tagged as "parameters" in papermill
# When executed via papermill, these defaults are overridden

# Parameters (defaults)
dataset_name = "bigquery-public-data.ml_datasets.penguins"
target_column = "species"
test_size = 0.2
random_state = 42

print(f"Dataset:       {dataset_name}")
print(f"Target column: {target_column}")
print(f"Test size:     {test_size}")
print(f"Random state:  {random_state}")

In [ ]:
# Demonstrate how to execute THIS notebook with papermill
import papermill as pm

# Example: execute with different parameters
# pm.execute_notebook(
#     "03-vertex-ai-notebooks.ipynb",   # input notebook
#     "output/run_penguins.ipynb",       # output notebook
#     parameters={
#         "dataset_name": "bigquery-public-data.ml_datasets.penguins",
#         "target_column": "species",
#         "test_size": 0.3,
#         "random_state": 123
#     }
# )

print("Papermill parameterized execution concept:")
print("  1. Tag a cell with 'parameters' containing default values")
print("  2. Call pm.execute_notebook() with override values")
print("  3. Papermill injects a new cell after the parameters cell")
print("  4. The output notebook contains all results")
print("")
print("On GCP, combine with Cloud Scheduler for scheduled runs:")
print("  Cloud Scheduler -> Cloud Function -> Papermill -> GCS output")

## 6. GPU Availability Check

Before training models, verify that GPU resources are available and properly configured.

In [ ]:
import subprocess
import shutil

print("=" * 60)
print("GPU AVAILABILITY CHECK")
print("=" * 60)

# Check NVIDIA GPU via nvidia-smi
if shutil.which("nvidia-smi"):
    print("\n--- NVIDIA GPU detected ---")
    result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    print(result.stdout)
else:
    print("\nNo NVIDIA GPU detected (nvidia-smi not found).")
    print("To use GPUs:")
    print("  - Colab Enterprise: Change runtime type to GPU")
    print("  - Workbench: Create instance with --accelerator-type flag")

In [ ]:
# Check GPU availability via TensorFlow
try:
    import tensorflow as tf
    gpus = tf.config.list_physical_devices('GPU')
    print(f"TensorFlow version: {tf.__version__}")
    print(f"GPUs available:     {len(gpus)}")
    for gpu in gpus:
        print(f"  - {gpu.name} ({gpu.device_type})")
    if not gpus:
        print("  No GPU available. Running on CPU.")
except ImportError:
    print("TensorFlow not installed.")

# Check GPU availability via PyTorch
try:
    import torch
    print(f"\nPyTorch version:    {torch.__version__}")
    print(f"CUDA available:     {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"CUDA version:       {torch.version.cuda}")
        print(f"GPU count:          {torch.cuda.device_count()}")
        for i in range(torch.cuda.device_count()):
            print(f"  - GPU {i}: {torch.cuda.get_device_name(i)}")
            mem = torch.cuda.get_device_properties(i).total_mem / (1024**3)
            print(f"    Memory: {mem:.1f} GB")
except ImportError:
    print("\nPyTorch not installed.")

## Summary

In this notebook, we covered:

1. **Runtime Environment Info** — How to check Python version, GCP project, CPU/RAM, and installed packages
2. **BigQuery Integration** — Query public datasets directly, estimate query costs with dry runs
3. **Cloud Storage** — Read/write files using pandas+gcsfs and the storage client library
4. **Custom Packages** — Install packages via pip, requirements.txt, and startup scripts
5. **Parameterized Execution** — Use Papermill to run notebooks with different parameters
6. **GPU Availability** — Check GPU resources via nvidia-smi, TensorFlow, and PyTorch

### Key Exam Points
- Colab Enterprise = serverless, collaborative, ephemeral runtimes
- Workbench Managed = Google-maintained, scheduled runs, auto-patching
- Workbench User-Managed = full VM control, sudo access, persistent disk
- Always use dedicated service accounts and configure idle shutdown